In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# 2. Preprocess dataset
ds = pd.read_csv("uber.csv")

ds.dropna(inplace=True)
ds.drop(columns=["Unnamed: 0", "key"], errors="ignore", inplace=True)

ds["pickup_datetime"] = pd.to_datetime(
    ds["pickup_datetime"], utc=True
)
ds["hour"] = ds["pickup_datetime"].dt.hour
ds.drop("pickup_datetime", axis=1, inplace=True)

In [3]:
# Remove invalid values
ds = ds[
    (ds["fare_amount"] > 0) &
    (ds["passenger_count"].between(1, 6)) &
    (ds["pickup_latitude"].between(40, 42)) &
    (ds["dropoff_latitude"].between(40, 42)) &
    (ds["pickup_longitude"].between(-75, -72)) &
    (ds["dropoff_longitude"].between(-75, -72))
]

In [4]:
# 3. Identify and remove fare outliers using IQR

Q1 = ds["fare_amount"].quantile(0.25)
Q3 = ds["fare_amount"].quantile(0.75)
IQR = Q3 - Q1

ds = ds[
    (ds["fare_amount"] >= Q1 - 1.5 * IQR) &
    (ds["fare_amount"] <= Q3 + 1.5 * IQR)
]

In [5]:
# 4. Check correlation

print(ds.corr(numeric_only=True)["fare_amount"])

fare_amount          1.000000
pickup_longitude     0.001255
pickup_latitude     -0.075714
dropoff_longitude    0.073951
dropoff_latitude    -0.086857
passenger_count      0.012437
hour                -0.012952
Name: fare_amount, dtype: float64


In [6]:
# 5. Split dataset
from sklearn.model_selection import train_test_split

X = ds.drop("fare_amount", axis=1)
y = ds["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# 6. Feature scaling for Linear Regression
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [8]:
# 7. Linear Regression
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

In [9]:
# 8. Random Forest Regression
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [10]:
# 9. Evaluate and compare models
from sklearn.metrics import r2_score, mean_squared_error

for name, pred in [
    ("Linear Regression", lr_pred),
    ("Random Forest", rf_pred)
]:
    print("\n", name)
    print("R2 Score:", r2_score(y_test, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))


 Linear Regression
R2 Score: 0.03158321675597586
RMSE: 4.07601689631466

 Random Forest
R2 Score: 0.7349103730558995
RMSE: 2.1325596558642044
